# Cameo Extraction — Assigning a Specific Agent

Demonstrates two ways to pin a job to a specific agent (or agent pool) when submitting an `@istari:extract` job against a Cameo `.mdzip` model.

| Option | Approach | When to use |
|--------|----------|-------------|
| **1** | Drop to `platform.client.add_job()` directly | One-off, no library changes needed |
| **2** | Extend `JobDefinition` with agent fields | Repeated use, keeps the fluent style |

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**.
- An agent with the **Cameo** integration and access to `@istari:extract`.
- A Cameo `.mdzip` file.

### Credentials

Create a `.env` file next to this notebook:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

## 1 · Connect

In [ ]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition, JobView

platform = IstariPlatform.from_env()

report = platform.client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(platform)

## 2 · Configure

In [ ]:
MDZIP_PATH       = Path.cwd() / "NCXTable-example.mdzip"
TOOL_VERSION     = "2024x-refresh2"
OPERATING_SYSTEM = "Windows 11"
DISPLAY_NAME     = "NCXTable-example-agent.mdzip"
EXTERNAL_ID      = "cameo-agent-assignment-demo"

# Set this to the agent ID you want to target (see Section 3 below).
# Leave as None to skip the agent-assigned jobs and just browse available agents.
TARGET_AGENT_ID  = None   # e.g. "agt-abc123"

assert MDZIP_PATH.exists(), f"Cameo file not found: {MDZIP_PATH}"
print(f"Model file:  {MDZIP_PATH}")
print(f"Agent ID:    {TARGET_AGENT_ID or '(not set — browse agents in Section 3 first)'}")

## 3 · Browse available agents

List agents that have the Cameo module loaded to find a valid `TARGET_AGENT_ID`.
Set `TARGET_AGENT_ID` in the config cell above, then re-run from Section 4 onwards.

In [ ]:
agents = platform.client.list_agents(
    module_name="dassault_cameo",
    size=25,
)

print(f"Agents with dassault_cameo: {agents.total}\n")
print(f"{'ID':<40} {'NAME':<30} {'OS':<20} {'STATUS'}")
print("-" * 110)
for a in agents.items:
    print(f"  {a.id:<38} {(a.name or ''):<30} {(a.host_os or ''):<20} {a.status_name}")

# Agents with status_name="Idle" are ready to accept a new job.
# Valid status values: Idle, ClaimingJob, ValidatingJob, ExecutingJob,
# UploadingJob, ExecutionFailed, ExecutionSuccess, Paused, Unknown

## 4 · Upload the Cameo model

In [ ]:
model = platform.upload_model(
    MDZIP_PATH,
    external_id=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
print(f"Uploaded model {model.id}")
print(model)

## 5 · Option 1 — assign via `platform.client.add_job()`

Bypass `JobDefinition` entirely and call `add_job()` on the raw client.
This exposes every parameter the API supports, including `assigned_agent_id`
and `assigned_agent_pool_id`.

After submission, wrap the returned `Job` in a `JobView` to get `.wait()`,
`.get_products()`, and the rest of the fluent interface.

In [ ]:
assert TARGET_AGENT_ID, "Set TARGET_AGENT_ID in the config cell before running this section."

# Submit directly — no JobDefinition needed.
job1_raw = platform.client.add_job(
    model_id=model.id,
    function="@istari:extract",
    tool_name="dassault_cameo",
    # tool_version=TOOL_VERSION,
    # operating_system=OPERATING_SYSTEM,
    assigned_agent_id=TARGET_AGENT_ID,
    # assigned_agent_pool_id="<pool-id>",  # alternative: assign to a pool
)

# Wrap in JobView to get the full fluent interface.
job1 = JobView(_job=job1_raw, _client=platform.client)
print(f"Submitted job {job1.id} assigned to agent {TARGET_AGENT_ID}; polling...")

job1.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 1 finished: {job1.status}")

products1 = job1.get_products()
print(f"Products: {[p.name for p in products1]}")

## 6 · Option 2 — extend `JobDefinition`

For repeated use, subclass `JobDefinition` to add the agent fields, then
provide a small helper that calls `add_job()` and returns a `JobView`.
This keeps the submission call site as clean as the standard fluent style
without patching the library.

If you need this across multiple notebooks, move the class and helper into
a shared module (e.g. `istari_fluent/istari_utils.py`) and wire
`assigned_agent_id` through `_submit_job_impl`.

In [ ]:
from pydantic import Field as PydanticField


class AgentJobDefinition(JobDefinition):
    """JobDefinition extended with agent/pool targeting."""
    assigned_agent_id: str | None = PydanticField(default=None)
    assigned_agent_pool_id: str | None = PydanticField(default=None)


def submit_job(platform: IstariPlatform, model_id: str, defn: AgentJobDefinition) -> JobView:
    """Submit an AgentJobDefinition and return a JobView ready to .wait()."""
    job_raw = platform.client.add_job(
        model_id=model_id,
        function=defn.function,
        tool_name=defn.tool_name,
        tool_version=defn.tool_version,
        operating_system=defn.operating_system,
        parameters=defn.build_parameters(),
        assigned_agent_id=defn.assigned_agent_id,
        assigned_agent_pool_id=defn.assigned_agent_pool_id,
    )
    return JobView(_job=job_raw, _client=platform.client)

In [ ]:
assert TARGET_AGENT_ID, "Set TARGET_AGENT_ID in the config cell before running this section."

extract_on_agent = AgentJobDefinition(
    function="@istari:extract",
    tool_name="dassault_cameo",
    # tool_version=TOOL_VERSION,
    # operating_system=OPERATING_SYSTEM,
    assigned_agent_id=TARGET_AGENT_ID,
    # assigned_agent_pool_id="<pool-id>",
)

job2 = submit_job(platform, model.id, extract_on_agent)
print(f"Submitted job {job2.id} assigned to agent {TARGET_AGENT_ID}; polling...")

job2.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 2 finished: {job2.status}")

products2 = job2.get_products()
print(f"Products: {[p.name for p in products2]}")

## Verify in the UI

1. **Jobs / Activity** — Both jobs should show `dassault_cameo / @istari:extract`.
2. **Job detail** — Each job should show the assigned agent under its execution details.

## Optional · Archive the model

In [ ]:
model.archive()